# Análise dos Resultados

Este notebook executa os experimentos do trabalho, carrega os resultados salvos em `results/` e produz gráficos para apoiar a discussão do relatório.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.experiments import run_kmeans_experiment, run_kmeans_sklearn_comparison, run_knn_experiment, run_knn_sklearn_comparison

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')
ROOT = Path('..').resolve()
TRAIN_FILE = ROOT / 'nba_treino.csv'
TEST_FILE = ROOT / 'nba_teste.csv'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
knn_custom = run_knn_experiment(TRAIN_FILE, TEST_FILE, [2, 10, 50, 7], RESULTS_DIR / 'knn')
knn_sklearn = run_knn_sklearn_comparison(TRAIN_FILE, TEST_FILE, [2, 10, 50, 7], RESULTS_DIR / 'knn_sklearn')
kmeans_custom = run_kmeans_experiment(TRAIN_FILE, TEST_FILE, [2, 3], RESULTS_DIR / 'kmeans')
kmeans_sklearn = run_kmeans_sklearn_comparison(TRAIN_FILE, TEST_FILE, [2, 3], RESULTS_DIR / 'kmeans_sklearn')

len(knn_custom), len(knn_sklearn), len(kmeans_custom), len(kmeans_sklearn)

In [ ]:
knn_rows = []
for row in knn_custom:
    knn_rows.append({**row, 'origin': 'custom'})
for row in knn_sklearn:
    knn_rows.append({**row, 'origin': 'sklearn'})

knn_df = pd.DataFrame(knn_rows)
knn_df[['origin', 'k', 'accuracy', 'precision', 'recall', 'f1']]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['accuracy', 'precision', 'recall', 'f1']
for ax, metric in zip(axes.flat, metrics):
    sns.barplot(data=knn_df, x='k', y=metric, hue='origin', ax=ax)
    ax.set_title(metric.upper())
    ax.set_ylim(0, 1)
    ax.legend(title='')
plt.tight_layout()

In [ ]:
for result in knn_custom:
    matrix = pd.DataFrame(result['confusion_matrix'], index=['true_0', 'true_1'], columns=['pred_0', 'pred_1'])
    display(matrix)
    plt.figure(figsize=(4, 3))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - k={result['k']}")
    plt.show()

In [ ]:
kmeans_rows = []
for row in kmeans_custom:
    kmeans_rows.append({'origin': 'custom', 'k': row['k'], 'clusters': row['cluster_summary']})
for row in kmeans_sklearn:
    kmeans_rows.append({'origin': 'sklearn', 'k': row['k'], 'clusters': row['cluster_summary']})
kmeans_rows

In [ ]:
for result in kmeans_custom:
    centroid_df = pd.DataFrame(result['centroids_scaled'])
    display(centroid_df)
    plt.figure(figsize=(12, 5))
    sns.heatmap(centroid_df, annot=False, cmap='coolwarm', center=0)
    plt.title(f"Centroides padronizados - k={result['k']}")
    plt.show()